# 13 — Aligned comparison of four representations (Phi-2)

**Research question:** how does outcome-conditioned Delta ID change when `finals`, `means`, `pre_last`, and `pre_mean` are compared using identical attempts and matched selections?

This notebook loads existing representation caches produced by notebooks 03 and 10. It does not load the model, generate responses, or extract new representations. The tokenizer is used to verify that the two extraction pipelines used comparable inputs. Input files are not modified.

- `finals`: the final token of the stored sequence, not necessarily the numeric answer.
- `means`: the mean over the stored sequence, including the prompt.
- `pre_last`: the token immediately before the token overlapping the last predicted numeric span selected by the extraction procedure.
- `pre_mean`: the mean over that prefix, including the prompt.

**Two explicit cohorts.** `common_all` contains all valid prefix attempts with original representations (2,576 were expected from the preceding audit). `common_clean` excludes the union of duplicate-group members across all four views and all layers after L2 normalization. Every member is excluded regardless of correctness. This is stricter than the earlier V3 audit of equality of complete pre_mean tensors. The exclusion count is therefore not forced to remain 35, nor the retained cohort to remain 2,541.

The primary aligned comparison uses `common_clean`. It describes a selected subset and does not validate or repair the original full-attempt result. If a problem loses a class, execution stops for review. `common_all` is estimated only for representations passing the audit at every layer; other views remain unestimated, without epsilon clipping. Exclusion effects can be described only where both estimates are available.

**Fixed settings:** the original LB mean-of-local-inverses formula; k=10 and 20; cap=10; row-wise L2 normalization; direct float64 Euclidean distances. Within a cohort, all four representations and both k values share matching and within-problem permutation plans.

**LOPO and pooled estimates share the same reference:** each problem is removed from EVERY pooled matched plan. LOPO Delta ID averages across those plans, just as the main pooled estimate does. Retained attempts are not resampled. This avoids relying on the single, atypically high reference draw used in the earlier V3 analysis. The optimized computation caches extra neighbors and was validated against full subset recomputation on synthetic data.

QUICK mode is enabled by default. Run numbered code cells 1-6 and review the audit, then run 7-10. For FINAL, set `QUICK=False` and run in order. Checkpoints allow resumption by completed representation, layer, and cohort.

**English edition:** explanations, comments, and display/error messages are translated. Analysis settings, data paths, and historical implementation identifiers are retained for checkpoint compatibility. Those identifiers refer to the original analysis version, not a byte hash of this translated file.


In [ ]:
# 1 — Imports. If packages are missing, install them in a separate cell:
# %pip install torch numpy pandas pyarrow matplotlib tqdm transformers
import json, hashlib, platform, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoTokenizer

try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/msc_thesis')
ORIGINAL_FILE = BASE / 'data/representations/phi2/phi2_repr_paired_v1.pt'
PREFIX_FILE = BASE / 'data/representations/phi2/phi2_preanswer_paired_v1.pt'
DATA_PATH = BASE / 'data/frozen/phi2_dataset_frozen_v1.parquet'
PAIRED_PATH = BASE / 'data/generations/phi2/phi2_run_M200_N20_seed42.paired_problem_ids.json'
AUDIT_PATH = BASE / 'results/preanswer_id_phi2/preanswer_locator_audit.csv'
QUICK = True
VIEWS = ['finals', 'means', 'pre_last', 'pre_mean']
K_VALUES = [10,20]
LAYERS = list(range(32))  # same layers for all cohorts and representations
CAP = 10
REPS_POOLED = 5 if QUICK else 120
REPS_PERM = 10 if QUICK else 80
RUN_COMMON_ALL_IF_VALID = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG = dict(version='aligned_v1', views=VIEWS, k_values=K_VALUES, layers=LAYERS,
    cap=CAP, normalize=True, dtype='float64', distance='direct cdist, no matmul',
    quick=QUICK, pooled_reps=REPS_POOLED, permutation_reps=REPS_PERM,
    matching_seed=1010, permutation_seed=2010,
    lopo='all matched plans, fixed retained attempts',
    cohort_rule='union of all exact normalized per-layer duplicate-group members across four views',
    common_all_if_valid=RUN_COMMON_ALL_IF_VALID)
for path in [ORIGINAL_FILE,PREFIX_FILE,DATA_PATH,PAIRED_PATH]:
    if not path.is_file():raise FileNotFoundError(path)
OUT_BASE = BASE / 'results/aligned_four_views_phi2_v1' / ('quick' if QUICK else 'final')
OUT_BASE.mkdir(parents=True, exist_ok=True)
print('Device:',DEVICE)
print(json.dumps(CONFIG,indent=2))

CONFIG['implementation_sha256'] = 'd0bd0621baa1239f5ef55830acc6e94179a812defe82338194d314aaf311da95'


In [ ]:
# 2 — Load and join by attempt identity, never by file row order.
def label_bool(x):
    if not isinstance(x,(bool,np.bool_,int,np.integer)) or x not in (0,1):
        raise ValueError(f'Invalid correctness label: {x!r}')
    return bool(x)

def load_index(path):
    # Trusted local caches: the original notebooks use the list[dict] format.
    rows = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(rows,list) or not rows:raise ValueError(f'Unexpected cache format: {path}')
    result = {}
    for r in rows:
        key = (int(r['problem_id']),int(r['attempt_id']))
        if key in result:raise ValueError(f'Duplicate attempt ID {key} in {path}')
        result[key] = r
    return result

original, prefix = load_index(ORIGINAL_FILE), load_index(PREFIX_FILE)
with PAIRED_PATH.open() as f: expected_pids = set(map(int,json.load(f)))
frozen = pd.read_parquet(DATA_PATH)
assert {'problem_id','attempt_id','correct','generated_text','prompt'} <= set(frozen.columns)
assert not frozen.duplicated(['problem_id','attempt_id']).any()
frozen = frozen.set_index(['problem_id','attempt_id'])
expected_keys = {tuple(map(int,key)) for key in frozen.index if int(key[0]) in expected_pids}
orig_keys = {key for key in original if key[0] in expected_pids}
pre_keys = {key for key in prefix if key[0] in expected_pids}
assert orig_keys == expected_keys, 'Original cache is incomplete for the frozen paired dataset'
assert pre_keys <= orig_keys and pre_keys, 'Prefix cache contains unaligned identities'
all_keys = sorted(pre_keys)
missing_prefix = sorted(orig_keys-pre_keys)
print('Original paired attempts:',len(orig_keys),'| Prefix paired attempts:',len(pre_keys))
print('Attempts without a valid prefix cache:',missing_prefix)
if AUDIT_PATH.is_file():
    audit = pd.read_csv(AUDIT_PATH)
    assert not audit.duplicated(['problem_id','attempt_id']).any()
    assert set(zip(audit.problem_id.astype(int),audit.attempt_id.astype(int))) == expected_keys
    flags = audit['ok'].astype(str).str.lower()
    assert flags.isin(['true','false','0','1']).all()
    valid = audit[flags.isin(['true','1'])]
    assert set(zip(valid.problem_id.astype(int),valid.attempt_id.astype(int))) == pre_keys
elif missing_prefix:
    raise FileNotFoundError('Missing locator audit to verify excluded attempts: '+str(AUDIT_PATH))

records = []
for key in tqdm(all_keys,desc='Alignment and tensor sanity'):
    a,b = original[key],prefix[key]
    label = label_bool(frozen.loc[key,'correct'])
    assert label_bool(a['correct']) == label_bool(b['correct']) == label, key
    assert 0 < int(b['prefix_T']) == int(b['answer_start_tok']) < int(b['T_full']) <= 512, key
    row = dict(problem_id=key[0],attempt_id=key[1],correct=label,
               T=int(a['T']),T_full=int(b['T_full']),prefix_T=int(b['prefix_T']))
    for view in VIEWS:
        x = (a if view in ['finals','means'] else b)[view]
        assert isinstance(x,torch.Tensor) and tuple(x.shape)==(32,2560), (key,view)
        assert torch.isfinite(x).all() and (x.double().norm(dim=1)>1e-12).all(), (key,view)
        row[view] = x
    records.append(row)
meta = pd.DataFrame([{k:r[k] for k in ['problem_id','attempt_id','correct','T','T_full','prefix_T']} for r in records])
assert set(meta.problem_id) == expected_pids
# Other cache fields (e.g. traj) are not required for this comparison.
del original
print(meta[['T','T_full','prefix_T']].describe())


In [ ]:
# 3 — Check that different special-token settings did not change the inputs.
# Load only the tokenizer; do not load the Phi-2 model.
tokenizer = AutoTokenizer.from_pretrained('microsoft/phi-2',use_fast=True)
assert tokenizer.is_fast
NUM_RE = re.compile(r'-?\d+\.?\d*')
token_rows = []
for key,r in tqdm(list(zip(all_keys,records)),desc='Tokenizer alignment audit'):
    text = str(frozen.loc[key,'generated_text'])
    prompt = str(frozen.loc[key,'prompt'])
    old_ids = tokenizer(text,truncation=True,max_length=512)['input_ids']
    enc = tokenizer(text,add_special_tokens=False,truncation=True,max_length=512,return_offsets_mapping=True)
    new_ids = enc['input_ids']
    matches = list(NUM_RE.finditer(text))
    if not matches:raise ValueError(f'No numeric span: {key}')
    match = matches[-1]
    overlapping = [j for j,(a,b) in enumerate(enc['offset_mapping']) if b>match.start() and a<match.end() and a!=b]
    cut_ok = bool(overlapping) and overlapping[0] == r['prefix_T']
    if text.startswith(prompt):
        in_continuation = match.start() >= len(prompt)
    else:
        prompt_len = len(tokenizer(prompt,add_special_tokens=False,truncation=True,max_length=512)['input_ids'])
        in_continuation = bool(overlapping) and overlapping[0] >= prompt_len
    stored = prefix[key]
    ok = (old_ids == new_ids and len(old_ids)==r['T'] and len(new_ids)==r['T_full']
          and cut_ok and in_continuation and str(stored['answer_text'])==match.group(0))
    token_rows.append(dict(problem_id=key[0],attempt_id=key[1],ok=ok,
        same_full_token_ids=old_ids==new_ids, old_length=len(old_ids), prefix_full_length=len(new_ids),
        expected_T=r['T'], expected_T_full=r['T_full'], cut_ok=cut_ok,
        prefix_token_hash=hashlib.sha256(np.asarray(new_ids[:r['prefix_T']],dtype=np.int64).tobytes()).hexdigest()))
token_audit = pd.DataFrame(token_rows)
display(token_audit[~token_audit.ok])
if not token_audit.ok.all():
    raise ValueError('STOP: extraction inputs are not comparable. Review the rows above before analysis.')
print('All inputs passed the token audit:',len(token_audit))
del prefix
# This audit checks metadata and tokenization alignment; it does not reproduce the forward pass.


In [ ]:
# 4 — Shared numerical and sampling functions
"""Aligned ID comparison: numerical and sampling core (NumPy, float64)."""
import hashlib
import numpy as np


def normalize_rows(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Expected a finite [N,D] array')
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    if (norm <= 1e-12).any():
        raise ValueError('Zero or near-zero input norm')
    return x / norm


def identical_groups(x):
    """Exact value equality, treating +0 and -0 as equal; no rounding/tolerance."""
    x = np.asarray(x, dtype=np.float64).copy()
    x[x == 0] = 0.0
    buckets = {}
    for i, row in enumerate(x):
        digest = hashlib.sha256(row.tobytes()).hexdigest()
        buckets.setdefault(digest, []).append(i)
    result = []
    for ids in buckets.values():
        if len(ids) > 1:
            if not all(np.array_equal(x[ids[0]], x[j]) for j in ids[1:]):
                raise RuntimeError('Unexpected fingerprint collision')
            result.append(ids)
    return result


def make_plans(problem_ids, labels, cap, reps, seed, permute=False):
    """All indices refer to the same cohort. A plan is shared by all views and k."""
    problem_ids, labels = np.asarray(problem_ids), np.asarray(labels, bool)
    if len(problem_ids) != len(labels) or cap < 1 or reps < 1:
        raise ValueError('Invalid matching inputs')
    rng = np.random.default_rng(seed)
    pools = []
    for pid in np.unique(problem_ids):
        c = np.flatnonzero((problem_ids == pid) & labels)
        i = np.flatnonzero((problem_ids == pid) & ~labels)
        count = min(cap, len(c), len(i))
        if count == 0:
            raise ValueError(f'Problem {pid} lost a class; cannot match')
        pools.append((pid, c, i, count))
    plans = []
    for _ in range(reps):
        C, I = [], []
        for pid, c, i, count in pools:
            cc = rng.choice(c, count, replace=False)
            ii = rng.choice(i, count, replace=False)
            if permute:
                both = rng.permutation(np.concatenate([cc, ii]))
                cc, ii = both[:count], both[count:]
            C.extend(cc); I.extend(ii)
        C, I = np.asarray(C, int), np.asarray(I, int)
        assert len(np.unique(np.r_[C, I])) == len(C) + len(I)
        assert np.array_equal(problem_ids[C], problem_ids[I])
        if not permute:
            assert labels[C].all() and (~labels[I]).all()
        plans.append((C, I))
    return plans


def local_lb(neighbors, k):
    """Original mean-of-local-inverses formula; no epsilon clipping/jitter/drops."""
    a = np.asarray(neighbors, dtype=np.float64)[..., :k]
    if k < 2 or a.shape[-1] != k or not np.isfinite(a).all() or (a <= 1e-12).any():
        raise FloatingPointError('Invalid/zero kNN distance or insufficient neighbors')
    denominator = np.log(a[..., -1:] / a[..., :-1]).mean(axis=-1)
    if not np.isfinite(denominator).all() or (denominator <= 1e-12).any():
        raise FloatingPointError('Degenerate LB denominator')
    local = 1.0 / denominator
    if not np.isfinite(local).all():
        raise FloatingPointError('Nonfinite local ID')
    return local


def class_statistics(D, indices, cohort_pids, all_pids, k_values, with_lopo=True):
    """Exact LOPO via extra cached neighbors, validated against full recomputation.

    Removing a problem removes at most m points from a class. Every surviving
    query's new first k neighbors must be among its original first k+m neighbors.
    Distances to other observations do not change under row-wise L2 normalization.
    """
    indices = np.asarray(indices, int)
    group = np.asarray(cohort_pids)[indices]
    all_pids = np.asarray(all_pids)
    sub = np.asarray(D[np.ix_(indices, indices)], dtype=np.float64)
    np.fill_diagonal(sub, np.inf)
    n, max_k = len(indices), max(k_values)
    max_removed = max(int((group == p).sum()) for p in all_pids)
    if n <= max_k or (with_lopo and n - max_removed <= max_k):
        raise ValueError('Insufficient pooled/LOPO support for requested k')
    needed = min(n - 1, max_k + (max_removed if with_lopo else 0))
    nn = np.argpartition(sub, needed - 1, axis=1)[:, :needed]
    distances = np.take_along_axis(sub, nn, axis=1)
    order = np.argsort(distances, axis=1, kind='stable')
    nn = np.take_along_axis(nn, order, axis=1)
    distances = np.take_along_axis(distances, order, axis=1)
    observed, diag = [], []
    for k in k_values:
        local = local_lb(distances, k)
        observed.append(local.mean())
        diag.append([local.min(), np.median(local), local.max()])
    lopo = np.empty((len(all_pids), len(k_values))) if with_lopo else None
    if with_lopo:
        neighbor_groups = group[nn]
        for j, pid in enumerate(all_pids):
            keep_queries = group != pid
            updated = np.where(neighbor_groups[keep_queries] == pid,
                               np.inf, distances[keep_queries])
            updated.sort(axis=1)
            for ki, k in enumerate(k_values):
                lopo[j, ki] = local_lb(updated, k).mean()
    return np.asarray(observed), lopo, np.asarray(diag)


def evaluate_layer(D, pids, pooled_plans, permutation_plans, k_values, progress=None):
    all_pids = np.unique(pids)
    nR, nB, nK, nP = len(pooled_plans), len(permutation_plans), len(k_values), len(all_pids)
    observed = np.empty((nR, 2, nK))
    lopo = np.empty((nR, nP, 2, nK))
    diagnostics = np.empty((nR, 2, nK, 3))
    iterator = enumerate(pooled_plans)
    if progress is not None:
        iterator = progress(iterator, total=nR, desc='Matched + LOPO', leave=False)
    for r, plan in iterator:
        for side, indices in enumerate(plan):
            try:
                obs, leave, diag = class_statistics(D, indices, pids, all_pids, k_values)
            except (ValueError, FloatingPointError) as exc:
                raise RuntimeError(f'pooled draw={r}, class={side}: {exc}') from exc
            observed[r, side] = obs
            lopo[r, :, side] = leave
            diagnostics[r, side] = diag
    permutation = np.empty((nB, 2, nK))
    iterator = enumerate(permutation_plans)
    if progress is not None:
        iterator = progress(iterator, total=nB, desc='Permutation', leave=False)
    for b, plan in iterator:
        for side, indices in enumerate(plan):
            try:
                permutation[b, side] = class_statistics(
                    D, indices, pids, all_pids, k_values, with_lopo=False)[0]
            except (ValueError, FloatingPointError) as exc:
                raise RuntimeError(f'permutation draw={b}, class={side}: {exc}') from exc
    return dict(observed=observed, lopo=lopo, permutation=permutation,
                diagnostics=diagnostics, problem_ids=all_pids)


def summarize_draws(x):
    x = np.asarray(x, np.float64)
    if not np.isfinite(x).all():
        raise FloatingPointError('Invalid values cannot be silently omitted')
    return x.mean(axis=0), np.percentile(x, 2.5, axis=0), np.percentile(x, 97.5, axis=0)


In [ ]:
# 5 — Duplicate audit and ONE shared cohort for all four representations.
# Audit the original common cohort before any matching.
all_pids = meta.problem_id.to_numpy(int)
all_labels = meta.correct.to_numpy(bool)
excluded_union = set()
exclusion_by_view = {view:set() for view in VIEWS}
duplicate_rows, audit_summary = [], []
for view in VIEWS:
    for layer in tqdm(LAYERS,desc=f'Duplicate audit {view}'):
        raw = np.stack([r[view][layer].double().numpy() for r in records])
        groups = identical_groups(normalize_rows(raw))
        for group_no,ids in enumerate(groups):
            excluded_union.update(ids); exclusion_by_view[view].update(ids)
            labels = all_labels[ids]
            for i in ids:
                duplicate_rows.append(dict(view=view,layer=layer,group_number=group_no,
                    row_index=i,problem_id=all_keys[i][0],attempt_id=all_keys[i][1],
                    correct=bool(all_labels[i]),group_size=len(ids),mixed_labels=bool(labels.any() and (~labels).any()),
                    crosses_problems=len(set(all_pids[ids]))>1))
        audit_summary.append(dict(view=view,layer=layer,n_duplicate_groups=len(groups),
                                  n_affected_attempts=sum(map(len,groups))))
clean_indices = np.array([i for i in range(len(records)) if i not in excluded_union],int)
if not len(clean_indices):raise ValueError('The audit excluded all attempts')
COHORTS = {'common_all':np.arange(len(records)), 'common_clean':clean_indices}
cohort_meta = meta.copy()
cohort_meta['excluded'] = [i in excluded_union for i in range(len(records))]
for view in VIEWS:cohort_meta['duplicate_in_'+view] = [i in exclusion_by_view[view] for i in range(len(records))]
display(pd.DataFrame([dict(view=v,affected_attempts=len(exclusion_by_view[v])) for v in VIEWS]))
print('Union of excluded attempts:',len(excluded_union),'| Remaining:',len(clean_indices))
display(cohort_meta[cohort_meta.excluded])
cohort_counts = []
for cohort,indices in COHORTS.items():
    for pid in sorted(expected_pids):
        rows = indices[all_pids[indices] == pid]
        nc = int(all_labels[rows].sum()); ni = len(rows)-nc
        cohort_counts.append(dict(cohort=cohort,problem_id=pid,n_correct=nc,n_incorrect=ni,n_matched=min(CAP,nc,ni)))
counts = pd.DataFrame(cohort_counts)
invalid = counts[(counts.n_correct==0)|(counts.n_incorrect==0)]
if len(invalid):
    display(invalid)
    raise ValueError('STOP: a problem lost a correctness class; problems are not silently dropped.')
for cohort,part in counts.groupby('cohort'):
    assert part.n_matched.sum()-part.n_matched.max() > max(K_VALUES), cohort
    print(cohort, 'problems=',len(part),'correct=',part.n_correct.sum(),'incorrect=',part.n_incorrect.sum(),
          'matched per class=',part.n_matched.sum())
ELIGIBLE_ALL = [v for v in VIEWS if not exclusion_by_view[v]]
print('Representations without duplicates across all layers in common_all:',ELIGIBLE_ALL)
# Do not expand the filter after the audit: stop if near-degenerate distances remain.


In [ ]:
# 6 — Plans, manifest, and paths for reproducible resumption.
def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

input_paths = [ORIGINAL_FILE,PREFIX_FILE,DATA_PATH,PAIRED_PATH]
if AUDIT_PATH.is_file():input_paths.append(AUDIT_PATH)
manifest = dict(config=CONFIG,inputs={str(p):sha256_file(p) for p in input_paths},
    tokenizer_commit=tokenizer.init_kwargs.get('_commit_hash'),
    versions=dict(python=platform.python_version(),numpy=np.__version__,torch=torch.__version__,pandas=pd.__version__),
    device=DEVICE,common_keys=all_keys,clean_indices=clean_indices.tolist(),
    eligible_common_all=ELIGIBLE_ALL)
RUN_SIGNATURE = hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
RES_DIR = OUT_BASE / RUN_SIGNATURE[:12]
RES_DIR.mkdir(parents=True,exist_ok=True)
(RES_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
cohort_meta.to_csv(RES_DIR/'cohort_membership.csv',index=False)
counts.to_csv(RES_DIR/'cohort_problem_counts.csv',index=False)
pd.DataFrame(audit_summary).to_csv(RES_DIR/'duplicate_audit_by_view_layer.csv',index=False)
pd.DataFrame(duplicate_rows).to_csv(RES_DIR/'duplicate_group_members.csv',index=False)
token_audit.to_csv(RES_DIR/'tokenization_audit.csv',index=False)
PLANS = {}
for cohort,indices in COHORTS.items():
    pids, labels = all_pids[indices],all_labels[indices]
    pooled = make_plans(pids,labels,CAP,REPS_POOLED,CONFIG['matching_seed'])
    perm = make_plans(pids,labels,CAP,REPS_PERM,CONFIG['permutation_seed'],permute=True)
    PLANS[cohort] = dict(pooled=pooled,permutation=perm)
    np.savez_compressed(RES_DIR/f'{cohort}_plans.npz',
        pooled=np.asarray(pooled),permutation=np.asarray(perm),cohort_keys=np.asarray(all_keys)[indices])
(RES_DIR/'checkpoints').mkdir(exist_ok=True)
print('Results and checkpoints:',RES_DIR)


## Computation and resumption

Distances are computed once per representation and layer, then reused across matching plans and omissions. LOPO removes both query points and neighbors belonging to the omitted problem. ID is estimated on the pooled cloud, not averaged across separate problem-level ID estimates.

Each checkpoint stores one completed representation/layer/cohort with an input and configuration signature. Rerunning code cell 7 skips validated completed checkpoints. After a kernel restart, run the preceding cells in order, then cell 7. Changes to inputs, QUICK/FINAL mode, or configuration produce a different directory. The calculation is not bit-identical to the old float32 code: the formula and target quantity are retained while the numerical procedure is aligned and stricter.

For `common_all`, a representation with duplicates at any layer is excluded from the entire estimation pipeline, even if an individual draw would not contain that duplicate pair. This is an explicit analysis-availability rule, not a claim that every draw is invalid.


In [ ]:
# 7 — Run: reuse the same distance matrix for both k values and all plans.
def checkpoint_path(cohort,view,layer):
    return RES_DIR/'checkpoints'/f'{cohort}__{view}__L{layer:02d}.npz'

def read_checkpoint(path):
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item()) != RUN_SIGNATURE:raise ValueError('Checkpoint signature mismatch')
        return {key:z[key].copy() for key in z.files if key!='signature'}

@torch.no_grad()
def distance_matrix(view,layer):
    # Normalization and auditing use the same NumPy float64 implementation.
    raw = np.stack([r[view][layer].double().numpy() for r in records])
    normalized = normalize_rows(raw)
    x = torch.from_numpy(normalized).to(DEVICE)
    D = torch.empty((len(x),len(x)),dtype=torch.float64,device='cpu')
    for start in range(0,len(x),256):
        block = torch.cdist(x[start:start+256],x,p=2,compute_mode='donot_use_mm_for_euclid_dist')
        D[start:start+len(block)] = block.cpu()
    matrix = D.numpy()
    np.fill_diagonal(matrix,np.inf)
    return matrix

for view in VIEWS:
    wanted = ['common_clean']
    if RUN_COMMON_ALL_IF_VALID and view in ELIGIBLE_ALL:wanted.append('common_all')
    for layer in tqdm(LAYERS,desc=view):
        pending = []
        for cohort in wanted:
            path = checkpoint_path(cohort,view,layer)
            if path.exists():
                read_checkpoint(path)  # validate the checkpoint before skipping it
            else:pending.append(cohort)
        if not pending:continue
        D_all = distance_matrix(view,layer)
        for cohort in pending:
            indices = COHORTS[cohort]
            D = D_all[np.ix_(indices,indices)]
            off_diagonal = ~np.eye(len(D),dtype=bool)
            bad = off_diagonal & ((D<=1e-12)|~np.isfinite(D))
            if bad.any():
                pairs = np.argwhere(np.triu(bad,1))[:10]
                print('Degenerate pairs:',[(all_keys[indices[a]],all_keys[indices[b]],D[a,b]) for a,b in pairs])
                raise FloatingPointError(f'{cohort} {view} L{layer}: invalid distances; the filter is not expanded automatically')
            try:
                values = evaluate_layer(D,all_pids[indices],PLANS[cohort]['pooled'],
                    PLANS[cohort]['permutation'],K_VALUES,progress=tqdm)
            except (ValueError,FloatingPointError,RuntimeError) as exc:
                raise RuntimeError(f'{cohort} {view} L{layer}: {exc}') from exc
            path = checkpoint_path(cohort,view,layer)
            temp = path.with_suffix('.tmp')
            with temp.open('wb') as f:np.savez_compressed(f,signature=np.asarray(RUN_SIGNATURE),**values)
            temp.replace(path)
        del D_all,D
        gc.collect()
print('All planned checkpoints are complete.')


In [ ]:
# 8 — Summarize all layers. The LOPO reference equals the pooled mean by construction.
RESULTS = {}
rows = []
for cohort in COHORTS:
    for view in VIEWS:
        eligible = cohort=='common_clean' or (RUN_COMMON_ALL_IF_VALID and view in ELIGIBLE_ALL)
        if not eligible:continue
        blocks = [read_checkpoint(checkpoint_path(cohort,view,ell)) for ell in LAYERS]
        # observed [draw,class,k,layer]; lopo [draw,omitted_problem,class,k,layer]
        observed = np.stack([b['observed'] for b in blocks],axis=-1)
        lopo = np.stack([b['lopo'] for b in blocks],axis=-1)
        permutation = np.stack([b['permutation'] for b in blocks],axis=-1)
        delta = observed[:,1]-observed[:,0]
        lopo_draw_delta = lopo[:,:,1]-lopo[:,:,0]
        lopo_mean_delta = lopo_draw_delta.mean(axis=0)
        perm_delta = permutation[:,1]-permutation[:,0]
        RESULTS[(cohort,view)] = dict(delta=delta,lopo=lopo_mean_delta,permutation=perm_delta)
        pm,plo,phi = summarize_draws(delta)
        rm,rlo,rhi = summarize_draws(perm_delta)
        omitted = blocks[0]['problem_ids']
        for ki,k in enumerate(K_VALUES):
            for li,layer in enumerate(LAYERS):
                j = lopo_mean_delta[:,ki,li]
                shift = j-pm[ki,li]
                worst = int(np.argmax(np.abs(shift)))
                rows.append(dict(cohort=cohort,representation=view,k=k,layer=layer,
                    pooled_id_correct=observed[:,0,ki,li].mean(),pooled_id_incorrect=observed[:,1,ki,li].mean(),
                    pooled_delta_id=pm[ki,li],sampling_p025=plo[ki,li],sampling_p975=phi[ki,li],
                    perm_delta_id=rm[ki,li],perm_p025=rlo[ki,li],perm_p975=rhi[ki,li],
                    lopo_reference=pm[ki,li],lopo_mean=j.mean(),lopo_min=j.min(),lopo_max=j.max(),
                    lopo_fraction_positive=(j>0).mean(),lopo_max_abs_shift=np.abs(shift).max(),
                    most_influential_problem=int(omitted[worst]),n_problems=len(omitted),
                    matched_per_class=len(PLANS[cohort]['pooled'][0][0]),reps_pooled=REPS_POOLED))
summary = pd.DataFrame(rows)
summary.to_csv(RES_DIR/'aligned_id_summary.csv',index=False)
primary = summary[summary.cohort=='common_clean']
display(primary[primary.layer.isin([0,8,16,24,31])][['representation','k','layer','pooled_delta_id',
    'sampling_p025','sampling_p975','perm_delta_id','lopo_reference','lopo_min','lopo_max','lopo_fraction_positive']].round(4))
# Summarize L24-L31 for every representation; keep this statistic separate from L31 alone.
late_rows = []
late_mask = np.array([24<=ell<=31 for ell in LAYERS])
for view in VIEWS:
    b = RESULTS[('common_clean',view)]
    late_draw = b['delta'][:,:,late_mask].mean(axis=2)
    mean,lo,hi = summarize_draws(late_draw)
    for ki,k in enumerate(K_VALUES):
        late_rows.append(dict(representation=view,k=k,layer_window='L24-L31',
            pooled_late_mean=mean[ki],sampling_p025=lo[ki],sampling_p975=hi[ki]))
late_summary = pd.DataFrame(late_rows)
late_summary.to_csv(RES_DIR/'aligned_late_layer_summary.csv',index=False)
display(late_summary.round(4))


In [ ]:
# 9 — Paired representation contrasts and descriptive effects of exclusion.
# Pair by the SAME matched draw, layer, and k within common_clean.
contrast_rows = []
for before,after in [('means','pre_mean'),('finals','pre_last')]:
    a,b = RESULTS[('common_clean',before)],RESULTS[('common_clean',after)]
    gap = b['delta']-a['delta']  # ΔID(prefix view) − ΔID(full view)
    gm,glo,ghi = summarize_draws(gap)
    jl = b['lopo']-a['lopo']
    for ki,k in enumerate(K_VALUES):
        for li,layer in enumerate(LAYERS):
            contrast_rows.append(dict(comparison=f'{after} minus {before}',k=k,layer=layer,
                paired_gap_mean=gm[ki,li],sampling_p025=glo[ki,li],sampling_p975=ghi[ki,li],
                lopo_gap_min=jl[:,ki,li].min(),lopo_gap_max=jl[:,ki,li].max()))
contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(RES_DIR/'paired_representation_contrasts.csv',index=False)
display(contrasts[contrasts.layer==31].round(4))

exclusion_rows = []
for view in VIEWS:
    for k in K_VALUES:
        for layer in LAYERS:
            clean = primary[(primary.representation==view)&(primary.k==k)&(primary.layer==layer)].iloc[0]
            original_rows = summary[(summary.cohort=='common_all')&(summary.representation==view)&(summary.k==k)&(summary.layer==layer)]
            available = len(original_rows)==1
            exclusion_rows.append(dict(representation=view,k=k,layer=layer,
                common_all_status='estimated' if available else ('not requested' if not RUN_COMMON_ALL_IF_VALID else 'duplicate audit prevents full pipeline'),
                common_all_delta_id=float(original_rows.iloc[0].pooled_delta_id) if available else None,
                common_clean_delta_id=float(clean.pooled_delta_id),
                clean_minus_all=float(clean.pooled_delta_id-original_rows.iloc[0].pooled_delta_id) if available else None))
exclusion_effect = pd.DataFrame(exclusion_rows)
exclusion_effect.to_csv(RES_DIR/'exclusion_effect_descriptive.csv',index=False)
display(exclusion_effect[exclusion_effect.layer==31].round(4))
# clean_minus_all is a descriptive cohort difference, not a paired-draw contrast or isolated causal effect.


In [ ]:
# 10 — Aligned plots with shared axes and explicit range descriptions; PNG output only.
colors = dict(finals='#264653',means='#2a9d8f',pre_last='#e76f51',pre_mean='#a15db5')
for k in K_VALUES:
    fig,ax = plt.subplots(figsize=(10,5))
    for view in VIEWS:
        s = primary[(primary.representation==view)&(primary.k==k)].sort_values('layer')
        x = s.layer.to_numpy()
        ax.plot(x,s.pooled_delta_id,label=view,color=colors[view])
        ax.fill_between(x,s.sampling_p025,s.sampling_p975,color=colors[view],alpha=.12)
    ax.axhline(0,color='black',ls='--',lw=1)
    ax.set(xlabel='Stored layer (0-based)',ylabel='ΔID = ID(I) − ID(C)',
        title=f'Common clean cohort | k={k} | '+('QUICK' if QUICK else 'FINAL'))
    ax.legend();ax.grid(alpha=.15)
    fig.text(.5,.01,'Shading: matched-draw 2.5–97.5 percentiles; not a confidence interval',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.04,1,1]);fig.savefig(RES_DIR/f'four_views_k{k}.png',dpi=180)
    plt.show();plt.close(fig)

    fig,axes = plt.subplots(2,2,figsize=(13,9),sharex=True,sharey=True)
    for ax,view in zip(axes.flat,VIEWS):
        s=primary[(primary.representation==view)&(primary.k==k)].sort_values('layer')
        x=s.layer.to_numpy()
        ax.plot(x,s.pooled_delta_id,label='Pooled = LOPO reference',color=colors[view])
        ax.fill_between(x,s.lopo_min,s.lopo_max,alpha=.2,color=colors[view],label='LOPO min–max; not CI')
        ax.plot(x,s.perm_delta_id,color='gray',ls='--',label='Permutation mean')
        ax.fill_between(x,s.perm_p025,s.perm_p975,color='gray',alpha=.1,label='Permutation 2.5–97.5%')
        ax.axhline(0,color='black',lw=.7);ax.set_title(view);ax.legend(fontsize=8);ax.grid(alpha=.15)
        ax.set_xlabel('Stored layer');ax.set_ylabel('ΔID')
    fig.suptitle(f'Common clean cohort | k={k} | LOPO across all matched plans')
    fig.tight_layout();fig.savefig(RES_DIR/f'four_views_lopo_perm_k{k}.png',dpi=180)
    plt.show();plt.close(fig)

    fig,ax=plt.subplots(figsize=(10,4))
    for comparison,s in contrasts[contrasts.k==k].groupby('comparison'):
        s=s.sort_values('layer');x=s.layer.to_numpy()
        ax.plot(x,s.paired_gap_mean,label=comparison)
        ax.fill_between(x,s.sampling_p025,s.sampling_p975,alpha=.15)
    ax.axhline(0,color='black',ls='--',lw=1)
    ax.set(xlabel='Stored layer',ylabel='Paired difference in ΔID',title=f'Same attempts and draws | k={k}')
    ax.legend();ax.grid(alpha=.15);fig.tight_layout()
    fig.savefig(RES_DIR/f'paired_representation_gaps_k{k}.png',dpi=180)
    plt.show();plt.close(fig)
print('Saved to:',RES_DIR)


## Interpretation and limitations

Positive prefix Delta ID describes an association between correctness and geometry before the selected numeric span in the common filtered cohort. It does not establish that the model knows the eventual outcome, exclude earlier numbers or answer content, or remove all length and style differences. Representation contrasts are not causal effects.

Pooled percentiles summarize attempt-selection variability. LOPO min-max ranges summarize problem omissions using the same averaging over matched plans. Permutation ranges describe individual balanced permuted samples, not the null distribution of the average across all pooled draws. No p-values or population confidence intervals are reported.

Filtering was motivated by an observed issue and was not preregistered. The union of exclusions aligns the comparison but may remove informative repeated prefixes, including groups with both outcomes. Report affected attempts, problems, and classes. `common_all` already requires a valid prefix and is not identical to the original thesis sample. Even estimable views do not constitute bit-identical reproduction of the historical notebook.

Auditing normalized equality cannot recover information lost when representations were stored in float16. The token audit checks stored-input and metadata compatibility with the available tokenizer; it does not establish the historical model version or reproduce extraction. The manifest records the available tokenizer commit.

**QUICK review artifacts:** `cohort_problem_counts.csv`, `duplicate_audit_by_view_layer.csv`, L31 rows from `aligned_id_summary.csv`, `paired_representation_contrasts.csv`, and overview plots. Full .pt caches are not needed for reviewing those exports. FINAL uses 120 pooled draws and 80 permutation draws, with LOPO across every problem and pooled plan.

**Validation scope:** the numerical core was tested on synthetic data, including both k values and optimized LOPO against full recomputation. This translated, output-free copy has not been rerun against the original Drive caches. Empirical results must be obtained from separately recorded runs.
